Как это работает:

    Агент 1 получает первоначальный запрос и готовит стартовые данные.

    Агент 2 анализирует данные, полученные от агента 1.

    Агент 3 формирует итоговый отчёт по результатам агента 2.

    Все агенты последовательно вызываются в рамках графа состояний LangGraph.

    Общение между агентами происходит через передачу сообщений (HumanMessage).


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import Tool, AgentExecutor
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, START
from langchain_core.messages import HumanMessage

# Инициализация LLM модели для всех агентов
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

# Определяем функции для каждого агента (могут быть сложные задачи)
def agent1_task(query: str) -> str:
    return f"Агент 1 обработал запрос: '{query}' и сообщил начальные данные."

def agent2_task(data: str) -> str:
    return f"Агент 2 получил данные: '{data}', выполнил анализ и предоставил вывод."

def agent3_task(summary: str) -> str:
    return f"Агент 3 получил вывод: '{summary}' и подготовил окончательный отчёт."

# Создаем инструменты для агентов
tool_agent1 = Tool(name="Agent1Tool", func=agent1_task, description="Обработка начальных данных.")
tool_agent2 = Tool(name="Agent2Tool", func=agent2_task, description="Анализ данных.")
tool_agent3 = Tool(name="Agent3Tool", func=agent3_task, description="Создание отчёта.")

# Мультиагентный граф, где агенты обмениваются сообщениями последовательно
workflow = StateGraph()

def call_agent1(state):
    input_message = state["messages"][-1].content  # Получаем последний запрос
    response_text = agent1_task(input_message)
    return {"messages": state["messages"] + [HumanMessage(response_text)]}

def call_agent2(state):
    last_msg = state["messages"][-1].content
    response_text = agent2_task(last_msg)
    return {"messages": state["messages"] + [HumanMessage(response_text)]}

def call_agent3(state):
    last_msg = state["messages"][-1].content
    response_text = agent3_task(last_msg)
    return {"messages": state["messages"] + [HumanMessage(response_text)]}

# Определяем последовательность вызовов агентов
workflow.add_edge(START, "agent1")
workflow.add_node("agent1", call_agent1)
workflow.add_edge("agent1", "agent2")
workflow.add_node("agent2", call_agent2)
workflow.add_edge("agent2", "agent3")
workflow.add_node("agent3", call_agent3)

# Компилируем граф
app = workflow.compile()

# Запускаем мультиагентный процесс с исходным запросом
initial_query = "Начните с анализа темы мультиагентных систем."
output = app.invoke({"messages": [HumanMessage(initial_query)]})

# Выводим поэтапные ответы агентов
for msg in output["messages"]:
    print(msg.content)


каждый агент отвечает за свою задачу и передает данные дальше через граф состояний

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

def agent1_task(state):
    query = state["messages"][-1].content
    response = f"Агент 1 обработал запрос: '{query}', подготовил данные."
    return {"messages": state["messages"] + [HumanMessage(response)]}

def agent2_task(state):
    prev = state["messages"][-1].content
    response = f"Агент 2 получил '{prev}', проанализировал данные."
    return {"messages": state["messages"] + [HumanMessage(response)]}

def agent3_task(state):
    prev = state["messages"][-1].content
    response = f"Агент 3 получил '{prev}', подготовил итоговый отчёт."
    return {"messages": state["messages"] + [HumanMessage(response)]}

workflow = StateGraph()

workflow.add_edge(START, "agent1")
workflow.add_node("agent1", agent1_task)
workflow.add_edge("agent1", "agent2")
workflow.add_node("agent2", agent2_task)
workflow.add_edge("agent2", "agent3")
workflow.add_node("agent3", agent3_task)

app = workflow.compile()

initial_query = "Начните с анализа темы мультиагентных систем."
output = app.invoke({"messages": [HumanMessage(initial_query)]})

for msg in output["messages"]:
    print(msg.content)
